# Notebook 10: Leave-One-Stock-Out Cross-Stock Generalisation

## Objective

This notebook tests whether machine-learning models can generalise to a **completely unseen stock**.

For each of the five stocks, that stock is held out for testing while the other four stocks are used for training.

- Training: **2020–2024**
- Test: **2025**

This is stricter than the pooled experiment because the held-out company is never present in training.

**Primary models:** Dummy Classifier, L2 Logistic Regression, Random Forest, and XGBoost.

The LSTM remains a supplementary baseline and is not repeated here to keep the main cross-stock experiment focused and manageable.

## Research Question

> **Can a model trained on multiple companies generalise to a completely unseen company?**

Example:

```text
BRK-B + CVX + GE + MSFT
          ↓
      train model
          ↓
       test NVDA
```

The procedure is repeated so every stock is held out once.

This directly addresses the cross-stock generalisation component of the research design.

In [3]:
# ==========================================================
# Imports
# ==========================================================

import random
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

RANDOM_STATE = 42
TEST_YEAR = 2025

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Imports completed.")

Imports completed.


## Load the Final Model-Ready Dataset

The same dataset used in Notebooks 08 and 09 is used so that only the experimental design changes.

In [4]:
# ==========================================================
# Load dataset
# ==========================================================

data_df = pd.read_csv(
    "/kaggle/input/notebooks/phyothaw/06-target-and-model-dataset-ipynb/model_ready_complete_case_2020_2025.csv"
)

data_df["date"] = pd.to_datetime(data_df["date"])

print(f"Dataset shape: {data_df.shape}")
print(f"Date range: {data_df['date'].min()} to {data_df['date'].max()}")

display(
    data_df["ticker"]
    .value_counts()
    .sort_index()
    .to_frame("Observations")
)

Dataset shape: (7370, 42)
Date range: 2020-01-13 00:00:00 to 2025-12-22 00:00:00


,Observations
ticker,
BRK-B,1474
CVX,1474
GE,1474
MSFT,1474
NVDA,1474


## Prepare GDELT Tone Features

In [5]:
# ==========================================================
# Missing GDELT tone handling
# ==========================================================

data_df["gdelt_tone_missing"] = (
    data_df["gdelt_company_tone"].isna().astype(int)
)

data_df["gdelt_company_tone"] = (
    data_df["gdelt_company_tone"].fillna(0)
)

remaining_missing = data_df.isna().sum()
display(
    remaining_missing[remaining_missing > 0].to_frame("Missing Count")
)

,Missing Count


## Feature Selection

The stock identifier, date, future-information columns, target columns, dividend/split columns, and the diagnostic `daily_return_check` are excluded.

In [6]:
# ==========================================================
# Feature selection
# ==========================================================

TARGET = "target_5d"

DROP_COLUMNS = [
    "ticker",
    "date",
    "future_close_5d",
    "forward_return_5d",
    "movement_5d",
    "target_5d",
    "Dividends",
    "Stock Splits",
    "daily_return_check"
]

FEATURE_COLUMNS = [
    c for c in data_df.columns
    if c not in DROP_COLUMNS
]

print(f"Number of predictor features: {len(FEATURE_COLUMNS)}")
print(FEATURE_COLUMNS)

Number of predictor features: 33
['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_50', 'SMA_200', 'EMA_50', 'EMA_200', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'ATR', 'ADX', 'gdelt_company_tone', 'gdelt_tone_missing', 'gdelt_tone_lag_1', 'gdelt_tone_mean_3', 'gdelt_tone_mean_5', 'gdelt_tone_std_5', 'gdelt_tone_change_1d', 'gdelt_abnormal_tone_20', 'market_close', 'market_return', 'vix_close', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']


## Temporal Data Preparation

Only observations before 2025 can enter training. The held-out stock's 2025 observations are used exclusively for testing.

In [7]:
# ==========================================================
# Temporal split
# ==========================================================

train_pool = data_df[
    data_df["date"].dt.year < TEST_YEAR
].copy()

test_2025 = data_df[
    data_df["date"].dt.year == TEST_YEAR
].copy()

stocks = sorted(data_df["ticker"].unique())

print("Stocks:", stocks)

print("\nTraining observations:")
display(
    train_pool["ticker"]
    .value_counts()
    .sort_index()
    .to_frame("Training Samples")
)

print("\n2025 test observations:")
display(
    test_2025["ticker"]
    .value_counts()
    .sort_index()
    .to_frame("Test Samples")
)

Stocks: ['BRK-B', 'CVX', 'GE', 'MSFT', 'NVDA']

Training observations:


,Training Samples
ticker,
BRK-B,1248
CVX,1248
GE,1248
MSFT,1248
NVDA,1248



2025 test observations:


,Test Samples
ticker,
BRK-B,226
CVX,226
GE,226
MSFT,226
NVDA,226


## Evaluation Function

In [8]:
# ==========================================================
# Common evaluation metrics
# ==========================================================

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "roc_auc_ovr": roc_auc_score(
            y_true, y_prob, multi_class="ovr", labels=[0, 1, 2]
        )
    }

## Leave-One-Stock-Out Experiment

For every held-out stock:

1. Train on the other four stocks using 2020–2024.
2. Test only on the held-out stock's 2025 observations.
3. Repeat for all five stocks.

Expected result:

**5 held-out stocks × 4 models = 20 evaluations.**

In [9]:
# ==========================================================
# Leave-one-stock-out experiment
# ==========================================================

all_results = []
all_predictions = []

for held_out_stock in stocks:

    print("=" * 70)
    print(f"Held-out stock: {held_out_stock}")
    print("=" * 70)

    train_df = train_pool[
        train_pool["ticker"] != held_out_stock
    ].copy()

    test_df = test_2025[
        test_2025["ticker"] == held_out_stock
    ].copy()

    train_df = train_df.sort_values(["ticker", "date"])
    test_df = test_df.sort_values("date")

    x_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET]

    x_test = test_df[FEATURE_COLUMNS]
    y_test = test_df[TARGET]

    training_stocks = sorted(train_df["ticker"].unique())

    print(f"Training stocks: {training_stocks}")
    print(f"Training rows: {len(train_df):,}")
    print(f"Test rows: {len(test_df):,}")

    # ------------------------------------------------------
    # Dummy
    # ------------------------------------------------------

    model = DummyClassifier(strategy="most_frequent")
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    prob = model.predict_proba(x_test)

    all_results.append({
        "held_out_stock": held_out_stock,
        "model": "Dummy Classifier",
        **evaluate_predictions(y_test, pred, prob)
    })

    all_predictions.append(pd.DataFrame({
        "held_out_stock": held_out_stock,
        "date": test_df["date"].values,
        "model": "Dummy Classifier",
        "actual": y_test.values,
        "predicted": pred,
        "prob_down": prob[:, 0],
        "prob_neutral": prob[:, 1],
        "prob_up": prob[:, 2]
    }))

    # ------------------------------------------------------
    # Random Forest
    # ------------------------------------------------------

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    prob = model.predict_proba(x_test)

    all_results.append({
        "held_out_stock": held_out_stock,
        "model": "Random Forest",
        **evaluate_predictions(y_test, pred, prob)
    })

    all_predictions.append(pd.DataFrame({
        "held_out_stock": held_out_stock,
        "date": test_df["date"].values,
        "model": "Random Forest",
        "actual": y_test.values,
        "predicted": pred,
        "prob_down": prob[:, 0],
        "prob_neutral": prob[:, 1],
        "prob_up": prob[:, 2]
    }))

    # ------------------------------------------------------
    # XGBoost
    # ------------------------------------------------------

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    prob = model.predict_proba(x_test)

    all_results.append({
        "held_out_stock": held_out_stock,
        "model": "XGBoost",
        **evaluate_predictions(y_test, pred, prob)
    })

    all_predictions.append(pd.DataFrame({
        "held_out_stock": held_out_stock,
        "date": test_df["date"].values,
        "model": "XGBoost",
        "actual": y_test.values,
        "predicted": pred,
        "prob_down": prob[:, 0],
        "prob_neutral": prob[:, 1],
        "prob_up": prob[:, 2]
    }))

    # ------------------------------------------------------
    # L2 Logistic Regression
    # ------------------------------------------------------

    scaler = StandardScaler()

    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    model = LogisticRegression(
        penalty="l2",
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    model.fit(x_train_scaled, y_train)

    pred = model.predict(x_test_scaled)
    prob = model.predict_proba(x_test_scaled)

    all_results.append({
        "held_out_stock": held_out_stock,
        "model": "Logistic Regression",
        **evaluate_predictions(y_test, pred, prob)
    })

    all_predictions.append(pd.DataFrame({
        "held_out_stock": held_out_stock,
        "date": test_df["date"].values,
        "model": "Logistic Regression",
        "actual": y_test.values,
        "predicted": pred,
        "prob_down": prob[:, 0],
        "prob_neutral": prob[:, 1],
        "prob_up": prob[:, 2]
    }))

print("\nExperiment completed.")

Held-out stock: BRK-B
Training stocks: ['CVX', 'GE', 'MSFT', 'NVDA']
Training rows: 4,992
Test rows: 226
Held-out stock: CVX
Training stocks: ['BRK-B', 'GE', 'MSFT', 'NVDA']
Training rows: 4,992
Test rows: 226
Held-out stock: GE
Training stocks: ['BRK-B', 'CVX', 'MSFT', 'NVDA']
Training rows: 4,992
Test rows: 226
Held-out stock: MSFT
Training stocks: ['BRK-B', 'CVX', 'GE', 'NVDA']
Training rows: 4,992
Test rows: 226
Held-out stock: NVDA
Training stocks: ['BRK-B', 'CVX', 'GE', 'MSFT']
Training rows: 4,992
Test rows: 226

Experiment completed.


## Cross-Stock Results

In [10]:
# ==========================================================
# Results table
# ==========================================================

loso_results_df = pd.DataFrame(all_results)

print(f"Number of evaluations: {len(loso_results_df)}")

assert len(loso_results_df) == 20, (
    "Expected 20 evaluations: 5 stocks × 4 models."
)

display(
    loso_results_df
    .sort_values(
        ["held_out_stock", "balanced_accuracy"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

Number of evaluations: 20


,held_out_stock,model,accuracy,balanced_accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr
0,BRK-B,Logistic Regression,0.4956,0.3570,0.5999,0.4956,0.3849,0.5403
1,BRK-B,Dummy Classifier,0.2788,0.3333,0.0777,0.2788,0.1215,0.5000
2,BRK-B,Random Forest,0.2168,0.3333,0.0479,0.2168,0.0784,0.5500
3,BRK-B,XGBoost,0.2168,0.3333,0.0470,0.2168,0.0773,0.5338
4,CVX,XGBoost,0.3319,0.3760,0.5170,0.3319,0.2616,0.5257
5,CVX,Dummy Classifier,0.3451,0.3333,0.1191,0.3451,0.1771,0.5000
6,CVX,Logistic Regression,0.2168,0.3333,0.0470,0.2168,0.0773,0.4715
7,CVX,Random Forest,0.3407,0.3291,0.1181,0.3407,0.1754,0.5505
8,GE,Logistic Regression,0.4071,0.3944,0.4445,0.4071,0.3942,0.6130
9,GE,Random Forest,0.3540,0.3783,0.4834,0.3540,0.3576,0.5256


## Average Performance Across Unseen Stocks

In [11]:
# ==========================================================
# Average model performance
# ==========================================================

loso_model_summary = (
    loso_results_df
    .groupby("model")[
        [
            "accuracy",
            "balanced_accuracy",
            "precision_weighted",
            "recall_weighted",
            "f1_weighted",
            "roc_auc_ovr"
        ]
    ]
    .mean()
    .sort_values("balanced_accuracy", ascending=False)
    .reset_index()
)

display(loso_model_summary)

,model,accuracy,balanced_accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr
0,Logistic Regression,0.4071,0.3584,0.3443,0.4071,0.3052,0.5344
1,XGBoost,0.3336,0.3544,0.3815,0.3336,0.2825,0.5295
2,Random Forest,0.3248,0.3431,0.2877,0.3248,0.2598,0.5236
3,Dummy Classifier,0.3752,0.3333,0.1475,0.3752,0.2098,0.5000


## Best Model for Each Held-Out Stock

In [12]:
# ==========================================================
# Best model per held-out stock
# ==========================================================

best_loso_per_stock = (
    loso_results_df
    .sort_values(
        ["held_out_stock", "balanced_accuracy"],
        ascending=[True, False]
    )
    .groupby("held_out_stock", as_index=False)
    .first()
)

display(
    best_loso_per_stock[
        [
            "held_out_stock",
            "model",
            "accuracy",
            "balanced_accuracy",
            "f1_weighted",
            "roc_auc_ovr"
        ]
    ]
)

,held_out_stock,model,accuracy,balanced_accuracy,f1_weighted,roc_auc_ovr
0,BRK-B,Logistic Regression,0.4956,0.3570,0.3849,0.5403
1,CVX,XGBoost,0.3319,0.3760,0.2616,0.5257
2,GE,Logistic Regression,0.4071,0.3944,0.3942,0.6130
3,MSFT,Logistic Regression,0.4602,0.3739,0.3840,0.5558
4,NVDA,XGBoost,0.4381,0.3610,0.3973,0.5646


## Balanced Accuracy by Held-Out Stock

In [13]:
# ==========================================================
# Stock × model comparison
# ==========================================================

loso_balanced_accuracy_matrix = (
    loso_results_df
    .pivot(
        index="held_out_stock",
        columns="model",
        values="balanced_accuracy"
    )
)

display(loso_balanced_accuracy_matrix)

model,Dummy Classifier,Logistic Regression,Random Forest,XGBoost
held_out_stock,,,,
BRK-B,0.3333,0.3570,0.3333,0.3333
CVX,0.3333,0.3333,0.3291,0.3760
GE,0.3333,0.3944,0.3783,0.3322
MSFT,0.3333,0.3739,0.3196,0.3694
NVDA,0.3333,0.3333,0.3551,0.3610


## Class Distribution Diagnostic

Cross-stock performance can be affected by differences in class proportions between the training stocks and the unseen test stock. This table records those differences.

In [14]:
# ==========================================================
# Class distribution diagnostic
# ==========================================================

class_rows = []

for held_out_stock in stocks:

    train_subset = train_pool[
        train_pool["ticker"] != held_out_stock
    ]

    test_subset = test_2025[
        test_2025["ticker"] == held_out_stock
    ]

    for class_value in [0, 1, 2]:

        train_share = (
            train_subset[TARGET] == class_value
        ).mean()

        test_share = (
            test_subset[TARGET] == class_value
        ).mean()

        class_rows.append({
            "held_out_stock": held_out_stock,
            "class": class_value,
            "training_class_share": train_share,
            "test_class_share": test_share,
            "share_difference": test_share - train_share
        })

class_distribution_df = pd.DataFrame(class_rows)

display(class_distribution_df)

,held_out_stock,class,training_class_share,test_class_share,share_difference
0,BRK-B,0,0.2987,0.2168,-0.0819
1,BRK-B,1,0.2815,0.5044,0.2230
2,BRK-B,2,0.4199,0.2788,-0.1411
3,CVX,0,0.2819,0.2168,-0.0650
4,CVX,1,0.3097,0.4381,0.1284
5,CVX,2,0.4085,0.3451,-0.0633
6,GE,0,0.2768,0.2257,-0.0512
7,GE,1,0.3327,0.2876,-0.0451
8,GE,2,0.3904,0.4867,0.0963
9,MSFT,0,0.2841,0.2522,-0.0318


## Held-Out Stock Isolation Check

This confirms that each held-out company is absent from its corresponding training set.

In [15]:
# ==========================================================
# Leakage / isolation check
# ==========================================================

isolation_rows = []

for held_out_stock in stocks:

    training_stocks = set(
        train_pool[
            train_pool["ticker"] != held_out_stock
        ]["ticker"].unique()
    )

    isolation_rows.append({
        "held_out_stock": held_out_stock,
        "present_in_training": held_out_stock in training_stocks
    })

isolation_check_df = pd.DataFrame(isolation_rows)

display(isolation_check_df)

assert not isolation_check_df["present_in_training"].any()

print("✓ All held-out stocks are absent from their training data.")

,held_out_stock,present_in_training
0,BRK-B,False
1,CVX,False
2,GE,False
3,MSFT,False
4,NVDA,False


✓ All held-out stocks are absent from their training data.


## Save Predictions and Results

Predicted probabilities are retained for the later reliability/calibration analysis.

In [16]:
# ==========================================================
# Combine predictions
# ==========================================================

loso_predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

probability_columns = [
    "prob_down",
    "prob_neutral",
    "prob_up"
]

probability_sum = (
    loso_predictions_df[probability_columns]
    .sum(axis=1)
)

print(
    "Maximum probability-sum error:",
    np.abs(probability_sum - 1).max()
)

print(
    f"Prediction rows: {len(loso_predictions_df):,}"
)

display(loso_predictions_df.head())

Maximum probability-sum error: 7.450580596923828e-08
Prediction rows: 4,520


,held_out_stock,date,model,actual,predicted,prob_down,prob_neutral,prob_up
0,BRK-B,2025-01-02,Dummy Classifier,0,2,0.0000,0.0000,1.0000
1,BRK-B,2025-01-03,Dummy Classifier,0,2,0.0000,0.0000,1.0000
2,BRK-B,2025-01-06,Dummy Classifier,1,2,0.0000,0.0000,1.0000
3,BRK-B,2025-01-07,Dummy Classifier,1,2,0.0000,0.0000,1.0000
4,BRK-B,2025-01-08,Dummy Classifier,2,2,0.0000,0.0000,1.0000


In [17]:
# ==========================================================
# Save outputs
# ==========================================================

loso_results_df.to_csv(
    "/kaggle/working/loso_model_performance.csv",
    index=False
)

loso_predictions_df.to_csv(
    "/kaggle/working/loso_model_predictions.csv",
    index=False
)

class_distribution_df.to_csv(
    "/kaggle/working/loso_class_distribution.csv",
    index=False
)

print("Saved:")
print("/kaggle/working/loso_model_performance.csv")
print("/kaggle/working/loso_model_predictions.csv")
print("/kaggle/working/loso_class_distribution.csv")

Saved:
/kaggle/working/loso_model_performance.csv
/kaggle/working/loso_model_predictions.csv
/kaggle/working/loso_class_distribution.csv


# Conclusion

Notebook 10 evaluates **cross-stock generalisation**.

The central question is whether models trained on four companies can predict a fifth company that was completely absent from training.

The results will be combined later with:

- **Notebook 08:** stock-specific modelling
- **Notebook 09:** pooled/global modelling
- **Notebook 10:** leave-one-stock-out generalisation

Notebook 11 will consolidate predictive-performance results rather than retraining these models.

Reliability measures such as calibration, Brier Score, Log Loss, and ECE are evaluated later.